# MAS: Multi-Agent System for Airport Anomaly Detection

This notebook implements an agentic workflow designed to detect and analyze security anomalies in airport passenger traffic. By utilizing a **Multi-Agent System (MAS)** built with `LangGraph` and local LLMs via `Ollama`, we separate data processing from high-level analytical reasoning.

### Key Objectives:
* **Contextual Data Ingestion**: Automatically filtering and cleaning passenger and alarm datasets based on user-defined perimeters.
* **Time-Series Math Integration**: Using classical statistical methods (Rolling Averages, Z-Scores) to anchor AI reasoning in mathematical reality.
* **Automated Situational Reporting**: Generating human-readable security reports that explain anomalies using both historical trends and demographic context.

In [44]:
import pandas as pd
import numpy as np
import operator
import io
import json  
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

## 2. Defining the Shared Memory (State)
In an agentic system, the **State** acts as the shared memory. It allows agents to pass structured data (like JSON strings) and natural language messages to one another.

* `perimeter`: The target airport IATA code (e.g., FCO, MXP, LIN).
* `passenger_json` / `alarm_json`: Cleaned datasets processed by the Data Agent.
* `baseline_stats`: The "Golden Standard" of math produced by the Baseline Agent.

In [45]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    perimeter: str
    passenger_json: str
    alarm_json: str
    baseline_dataframe_json: str
    anomaly_results: str
    risk_profile: str


## 3. Data Extraction & Normalization

### Italian Date Normalization (`clean_italian_dates`)
One of the primary challenges in this dataset is the presence of non-standard Italian date formats (e.g., "GEN" for January). 
The `clean_italian_dates` function performs the following:
1.  **Translation**: Maps Italian month abbreviations (GEN, MAG, DIC) to English equivalents.
2.  **Robust Parsing**: Uses Pandas' `mixed` format parsing with `dayfirst=True` to ensure consistency across different CSV formats.


In [46]:
import re

def clean_italian_dates(series):
    # mapping italian abbreviations to english 
    month_map = {
        'GEN': 'JAN', 'FEB': 'FEB', 'MAR': 'MAR', 'APR': 'APR',
        'MAG': 'MAY', 'GIU': 'JUN', 'LUG': 'JUL', 'AGO': 'AUG',
        'SET': 'SEP', 'OTT': 'OCT', 'NOV': 'NOV', 'DIC': 'DEC'
    }
    
    def replace_month(date_str):
        if pd.isna(date_str): return date_str
        date_str = str(date_str).upper()
        for it, en in month_map.items():
            if it in date_str:
                return date_str.replace(it, en)
        return date_str

    # apply translation and then parse with mixed format
    translated = series.apply(replace_month)
    return pd.to_datetime(translated, format='mixed', dayfirst=True, errors='coerce')

### Security Context Fetching (`fetch_security_context`)
This function acts as the "Retriever" for our system. It filters the raw CSVs based on the user's input perimeter, cleans numeric columns (handling missing or malformed data), and prepares the dataframes for analysis.

In [47]:
def fetch_security_context(perimeter: str):
    p_up = perimeter.upper().strip()
    
    # passengers data
    df_p = pd.read_csv('io/TIPOLOGIA_VIAGGIATORE.csv', sep=None, engine='python')
    mask_p = (df_p['AREOPORTO_ARRIVO'].str.upper() == p_up) | \
             (df_p['CITTA_ARR'].str.upper().str.contains(p_up, na=False))
    p_df = df_p[mask_p].copy()
    
    # numeric and dates cleaning
    for col in ['ENTRATI', 'INVESTIGATI', 'ALLARMATI']:
        p_df[col] = pd.to_numeric(p_df[col], errors='coerce').fillna(0).astype(int)
    
    p_df['timestamp'] = clean_italian_dates(p_df['DATA_PARTENZA'])
    p_df = p_df.dropna(subset=['timestamp']) # Remove unparseable dates
    p_df['join_date'] = p_df['timestamp'].dt.date

    # alarms data
    df_a = pd.read_csv('io/ALLARMI.csv', sep=None, engine='python')
    mask_a = (df_a['AREOPORTO_ARRIVO'].str.upper() == p_up) | \
             (df_a['CITTA_ARR'].str.upper().str.contains(p_up, na=False))
    a_df = df_a[mask_a].copy()
    
    if 'TOT' in a_df.columns:
        a_df['TOT'] = pd.to_numeric(a_df['TOT'], errors='coerce').fillna(0).astype(int)

    a_df['timestamp'] = clean_italian_dates(a_df['DATA_PARTENZA'])
    a_df = a_df.dropna(subset=['timestamp'])
    a_df['join_date'] = a_df['timestamp'].dt.date
    
    return p_df, a_df

## 4. Analytical Tools: The Anti-Hallucination Engine

### Statistical Logic (`calculate_security_stats`)
LLMs often struggle with precise mathematical calculations over large datasets, sometimes hallucinating impossible percentages. To solve this, we implement a **Classical Math Tool**. 

This function calculates:
* **7-Observation Rolling Averages**: Handles irregular data gaps by looking at the last 7 actual data points rather than calendar days.
* **Deviation Ratios**: Compares current activity to the recent 7-point average to identify sudden spikes.
* **Monthly Z-Scores**: Standardizes traffic volume against the historical monthly mean to account for seasonal variations.
* **National Demographics**: Summarizes the top traveler nationalities involved in the latest observations.

In [48]:
def build_baseline_dataframe(passenger_data: str, alarm_data: str) -> pd.DataFrame:
    """Build the row-level baseline table that downstream agents can inspect."""
    p_df = pd.read_json(io.StringIO(passenger_data), orient='records')
    a_df = pd.read_json(io.StringIO(alarm_data), orient='records')

    if p_df.empty:
        return pd.DataFrame(columns=[
            'date', 'entries', 'investigations', 'alarms', 'alarm_rate', 'inv_rate',
            'entries_7_mean', 'alarm_rate_7_mean', 'entries_dev_ratio',
            'alarm_dev_ratio', 'month', 'monthly_mean', 'monthly_std',
            'entries_z_score'
        ])

    p_df['timestamp'] = pd.to_datetime(p_df['timestamp'], errors='coerce')
    p_df = p_df.dropna(subset=['timestamp'])

    if not a_df.empty:
        a_df['timestamp'] = pd.to_datetime(a_df['timestamp'], errors='coerce')
        a_df = a_df.dropna(subset=['timestamp'])
    else:
        a_df = pd.DataFrame(columns=['timestamp', 'OCCORRENZE', 'TOT'])

    # 1. Aggregate daily passenger observations
    daily_pass = (
        p_df.groupby(p_df['timestamp'].dt.date)
        .agg({'ENTRATI': 'sum', 'INVESTIGATI': 'sum'})
        .reset_index()
    )
    daily_pass.columns = ['date', 'entries', 'investigations']

    # 2. Aggregate real alarms only
    if not a_df.empty and 'OCCORRENZE' in a_df.columns and 'TOT' in a_df.columns:
        real_alarms = a_df[a_df['OCCORRENZE'].astype(str).str.contains('Allarmi', na=False, case=False)]
        daily_alarms = (
            real_alarms.groupby(real_alarms['timestamp'].dt.date)['TOT']
            .sum()
            .reset_index()
        )
        daily_alarms.columns = ['date', 'alarms']
    else:
        daily_alarms = pd.DataFrame(columns=['date', 'alarms'])

    # 3. Merge into one timeline
    df = pd.merge(daily_pass, daily_alarms, on='date', how='left').fillna(0)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # 4. Rates
    safe_entries = df['entries'].replace(0, 1)
    df['alarm_rate'] = df['alarms'] / safe_entries
    df['inv_rate'] = df['investigations'] / safe_entries

    # 5. Rolling baselines
    df['entries_7_mean'] = df['entries'].rolling(window=7, min_periods=1).mean()
    df['alarm_rate_7_mean'] = df['alarm_rate'].rolling(window=7, min_periods=1).mean()

    # 6. Deviation features for outlier detection
    df['entries_dev_ratio'] = df['entries'] / df['entries_7_mean'].replace(0, 1)
    df['alarm_dev_ratio'] = df['alarm_rate'] / df['alarm_rate_7_mean'].replace(0, 1)

    # 7. Seasonal monthly baseline
    df['month'] = df['date'].dt.month
    monthly_stats = (
        df.groupby('month')['entries']
        .agg(['mean', 'std'])
        .reset_index()
        .rename(columns={'mean': 'monthly_mean', 'std': 'monthly_std'})
    )

    df = pd.merge(df, monthly_stats, on='month', how='left')
    df['monthly_std'] = df['monthly_std'].fillna(1).replace(0, 1)
    df['entries_z_score'] = (df['entries'] - df['monthly_mean']) / df['monthly_std']

    return df


def calculate_security_stats(passenger_data: str, alarm_data: str):
    """Return a compact baseline summary for reporting/logging."""
    baseline_df = build_baseline_dataframe(passenger_data, alarm_data)

    p_df = pd.read_json(io.StringIO(passenger_data), orient='records')
    if p_df.empty or baseline_df.empty:
        return {
            "latest_observation_date": None,
            "current_entries": 0,
            "monthly_z_score": 0.0,
            "current_alarm_rate": "0.0%",
            "alarm_deviation_ratio": 0.0,
            "7_obs_historical_alarm_rate": "0.0%",
            "top_nationalities": "No data available"
        }

    latest = baseline_df.iloc[-1]

    nat_counts = p_df['NAZIONALITA'].astype(str).value_counts(normalize=True).head(3)
    nat_summary = [f"{k} ({round(v * 100, 1)}%)" for k, v in nat_counts.items()]

    return {
        "latest_observation_date": str(latest['date'].date()),
        "current_entries": int(latest['entries']),
        "monthly_z_score": round(float(latest['entries_z_score']), 2),
        "current_alarm_rate": f"{round(float(latest['alarm_rate']) * 100, 2)}%",
        "alarm_deviation_ratio": round(float(latest['alarm_dev_ratio']), 2),
        "7_obs_historical_alarm_rate": f"{round(float(latest['alarm_rate_7_mean']) * 100, 2)}%",
        "top_nationalities": ", ".join(nat_summary)
    }


## 

## 5. AI Agent Nodes

### Data Agent (`data_agent_node`)
This agent is responsible for intent recognition. It uses the LLM to extract a clean IATA code from user input and triggers the data ingestion pipeline. It serves as the entry point for the graph.


In [49]:
def data_agent_node(state: AgentState):
    llm = ChatOllama(model="llama3.2:3b", temperature=0)
    prompt = f"Extract ONLY the 3-letter IATA code from: '{state['messages'][-1].content}'. No words, just the code (e.g. FCO)."
    perimeter = llm.invoke([HumanMessage(content=prompt)]).content.strip().upper()
    
    # regex safety to ensure we have a clean code
    perimeter = re.findall(r'[A-Z]{3}', perimeter)[0] if re.findall(r'[A-Z]{3}', perimeter) else perimeter
    
    p_df, a_df = fetch_security_context(perimeter)
    print(f"DATA AGENT: {perimeter} ({len(p_df)} records)")

    return {
        "perimeter": perimeter,
        "passenger_json": p_df.to_json(orient='records', date_format='iso'),
        "alarm_json": a_df.to_json(orient='records', date_format='iso')
    }

from langchain_core.tools import tool



### Baseline Agent (`baseline_agent_node`)
Acting as a **Senior Border Security Analyst**, this agent:
1.  Calls the math tools to get precise metrics.
2.  Synthesizes the math with demographic context.
3.  Writes a neutral, objective situational report that avoids bias while highlighting significant statistical deviations.

In [50]:
@tool
def security_math_tool(passenger_json: str, alarm_json: str):
    """Use this tool to calculate precise security statistics and anomaly thresholds."""
    return calculate_security_stats(passenger_json, alarm_json)

def baseline_agent_node(state: AgentState):
    print("BASELINE AGENT: Preparing Data Hand-off for Outlier Agent")

    baseline_df = build_baseline_dataframe(state["passenger_json"], state["alarm_json"])
    baseline_dataframe_json = baseline_df.to_json(orient='records', date_format='iso')

    report_msg = HumanMessage(
        content=(
            f"Baseline dataframe generated for {state['perimeter']} "
            f"with {len(baseline_df)} daily rows. Ready for Outlier Detection."
        )
    )

    return {
        "baseline_dataframe_json": baseline_dataframe_json,
        "messages": [report_msg]
    }


## 6. Data Integrity Check
Before running the graph, we perform a quick audit of the source files to ensure we have valid coverage for the major Italian perimeters (NAP, FCO, LIN, MXP, etc.). This ensures the system has a valid target before the agents begin processing.

In [51]:

df_audit = pd.read_csv('io/TIPOLOGIA_VIAGGIATORE.csv', sep=None, engine='python')

print("Unique Airport Codes in CSV:")
print(df_audit['AREOPORTO_ARRIVO'].unique()[:10]) # Show first 10

print("\nUnique Cities in CSV:")
print(df_audit['CITTA_ARR'].unique()[:10])

Unique Airport Codes in CSV:
<StringArray>
['NAP', 'FCO', 'TSF', 'BGY', 'PSA', 'BLQ', 'MXP', 'LIN', 'VRN', 'BRI']
Length: 10, dtype: str

Unique Cities in CSV:
<StringArray>
[ 'Napoli',    'Roma', 'Treviso', 'Bergamo',    'Pisa', 'Bologna',  'Milano',
  'Verona',    'Bari', 'Catania']
Length: 10, dtype: str


In [52]:
def validate_baseline_output(passenger_json: str, alarm_json: str, verbose: bool = True):
    """
    Validates that the baseline dataframe is:
    1. built correctly
    2. serialized correctly
    3. complete enough for the Outlier Agent
    4. numerically sane
    """

    print("=== BUILDING BASELINE DATAFRAME ===")
    baseline_df = build_baseline_dataframe(passenger_json, alarm_json)

    # -----------------------------
    # 1. Basic structure checks
    # -----------------------------
    print("\n[1] BASIC STRUCTURE")
    print("Shape:", baseline_df.shape)
    print("Columns:", list(baseline_df.columns))
    print("\nHead:")
    print(baseline_df.head())

    assert not baseline_df.empty, "Baseline dataframe is empty."

    required_cols = [
        'date',
        'entries',
        'investigations',
        'alarms',
        'alarm_rate',
        'inv_rate',
        'entries_7_mean',
        'alarm_rate_7_mean',
        'entries_dev_ratio',
        'alarm_dev_ratio',
        'month',
        'monthly_mean',
        'monthly_std',
        'entries_z_score'
    ]

    missing_cols = [col for col in required_cols if col not in baseline_df.columns]
    assert len(missing_cols) == 0, f"Missing required columns: {missing_cols}"
    print("Required columns check: OK")

    # -----------------------------
    # 2. Null / type checks
    # -----------------------------
    print("\n[2] NULL CHECK")
    null_counts = baseline_df[required_cols].isnull().sum()
    print(null_counts)

    if null_counts.sum() > 0:
        print("\nWarning: Null values found. Filling numeric nulls with 0.")
        numeric_cols = baseline_df.select_dtypes(include='number').columns
        baseline_df[numeric_cols] = baseline_df[numeric_cols].fillna(0)

    # -----------------------------
    # 3. Sanity checks
    # -----------------------------
    print("\n[3] SANITY CHECKS")

    assert (baseline_df['entries'] >= 0).all(), "Negative values found in 'entries'."
    assert (baseline_df['investigations'] >= 0).all(), "Negative values found in 'investigations'."
    assert (baseline_df['alarms'] >= 0).all(), "Negative values found in 'alarms'."
    assert (baseline_df['alarm_rate'] >= 0).all(), "Negative values found in 'alarm_rate'."
    assert (baseline_df['inv_rate'] >= 0).all(), "Negative values found in 'inv_rate'."

    print("Min/Max checks:")
    print("entries:", baseline_df['entries'].min(), "->", baseline_df['entries'].max())
    print("alarms:", baseline_df['alarms'].min(), "->", baseline_df['alarms'].max())
    print("alarm_rate:", baseline_df['alarm_rate'].min(), "->", baseline_df['alarm_rate'].max())
    print("entries_z_score:", baseline_df['entries_z_score'].min(), "->", baseline_df['entries_z_score'].max())

    # -----------------------------
    # 4. JSON serialization test
    # -----------------------------
    print("\n[4] SERIALIZATION TEST")
    baseline_dataframe_json = baseline_df.to_json(orient='records', date_format='iso')
    print("Serialized JSON preview:")
    print(baseline_dataframe_json[:500])

    reconstructed_df = pd.read_json(io.StringIO(baseline_dataframe_json), orient='records')

    print("\nReconstructed shape:", reconstructed_df.shape)
    print("Reconstructed columns:", list(reconstructed_df.columns))

    assert reconstructed_df.shape[0] == baseline_df.shape[0], "Row count changed after serialization."
    assert set(reconstructed_df.columns) == set(baseline_df.columns), "Columns changed after serialization."

    print("Serialization/deserialization check: OK")

    # -----------------------------
    # 5. Compare one sample day with raw data
    # -----------------------------
    print("\n[5] MANUAL SAMPLE-DAY CHECK")

    p_df = pd.read_json(io.StringIO(passenger_json), orient='records')
    a_df = pd.read_json(io.StringIO(alarm_json), orient='records')

    if not p_df.empty:
        p_df['timestamp'] = pd.to_datetime(p_df['timestamp'], errors='coerce')
        p_df = p_df.dropna(subset=['timestamp'])
        p_df['date'] = p_df['timestamp'].dt.date

    if not a_df.empty:
        a_df['timestamp'] = pd.to_datetime(a_df['timestamp'], errors='coerce')
        a_df = a_df.dropna(subset=['timestamp'])
        a_df['date'] = a_df['timestamp'].dt.date

    sample_row = baseline_df.iloc[0]
    sample_date = sample_row['date'].date()

    print("Sample date:", sample_date)

    raw_entries = 0
    raw_investigations = 0
    raw_alarms = 0

    if not p_df.empty:
        raw_entries = p_df.loc[p_df['date'] == sample_date, 'ENTRATI'].sum()
        raw_investigations = p_df.loc[p_df['date'] == sample_date, 'INVESTIGATI'].sum()

    if not a_df.empty and 'OCCORRENZE' in a_df.columns and 'TOT' in a_df.columns:
        raw_alarm_rows = a_df[
            (a_df['date'] == sample_date) &
            (a_df['OCCORRENZE'].astype(str).str.contains('Allarmi', case=False, na=False))
        ]
        raw_alarms = raw_alarm_rows['TOT'].sum()

    print("Raw entries:", raw_entries, "| Baseline entries:", sample_row['entries'])
    print("Raw investigations:", raw_investigations, "| Baseline investigations:", sample_row['investigations'])
    print("Raw alarms:", raw_alarms, "| Baseline alarms:", sample_row['alarms'])

    assert int(raw_entries) == int(sample_row['entries']), "Entries mismatch on sample day."
    assert int(raw_investigations) == int(sample_row['investigations']), "Investigations mismatch on sample day."
    assert int(raw_alarms) == int(sample_row['alarms']), "Alarms mismatch on sample day."

    print("Sample-day aggregation check: OK")

    # -----------------------------
    # 6. Final report
    # -----------------------------
    print("\n=== VALIDATION PASSED ===")
    print("Baseline dataframe is structurally correct and ready for the Outlier Agent.")

    return {
        "baseline_df": baseline_df,
        "baseline_json": baseline_dataframe_json,
        "reconstructed_df": reconstructed_df
    }

## 6.5 Outlier Detection & Risk Profiling Agents

Two additional agents extend the pipeline beyond the baseline computation:

- **Outlier Detection Agent** (`phi4-mini-reasoning`): reads the baseline dataframe, runs three unsupervised methods (IsolationForest, LOF, Z-score), takes a consensus of at least 2 out of 3, and lets the LLM interpret the flagged rows.
- **Risk Profiling Agent** (`qwen2.5:7b`): evaluates each flagged observation against four risk indicators and assigns a risk level (CRITICAL, HIGH, MODERATE, LOW) with a one-sentence rationale.

In [53]:
def run_isolation_forest(df, features, contamination=0.05):
    X = StandardScaler().fit_transform(df[features].fillna(0).values)
    iso = IsolationForest(contamination=contamination, random_state=42)
    labels = iso.fit_predict(X)
    return {
        "method": "IsolationForest",
        "flagged": [int(i) for i in np.where(labels == -1)[0]],
        "scores": iso.decision_function(X).tolist()
    }

def run_lof(df, features, contamination=0.05):
    X = StandardScaler().fit_transform(df[features].fillna(0).values)
    lof = LocalOutlierFactor(n_neighbors=min(20, len(df)-1), contamination=contamination)
    labels = lof.fit_predict(X)
    return {
        "method": "LOF",
        "flagged": [int(i) for i in np.where(labels == -1)[0]],
        "scores": lof.negative_outlier_factor_.tolist()
    }

def run_zscore(df, features, threshold=3.0):
    X = StandardScaler().fit_transform(df[features].fillna(0).values)
    max_z = np.abs(X).max(axis=1)
    return {
        "method": "Z-score",
        "flagged": [int(i) for i in np.where(max_z > threshold)[0]],
        "scores": max_z.tolist()
    }


def outlier_detection_agent_node(state):
    print("OUTLIER DETECTION AGENT")

    df = pd.read_json(io.StringIO(state["baseline_dataframe_json"]), orient="records")
    df["date"] = pd.to_datetime(df["date"])

    if len(df) < 5:
        result = {"error": f"Only {len(df)} rows — not enough for detection"}
        return {
            "anomaly_results": json.dumps(result),
            "messages": [HumanMessage(content=f"Outlier Detection skipped: {len(df)} rows")]
        }

    features = ["entries", "investigations", "alarms", "alarm_rate", "inv_rate",
                "entries_7_mean", "alarm_rate_7_mean",
                "entries_dev_ratio", "alarm_dev_ratio", "entries_z_score"]
    features = [f for f in features if f in df.columns]

    iso = run_isolation_forest(df, features)
    lof = run_lof(df, features)
    z = run_zscore(df, features)

    iso_set, lof_set, z_set = set(iso["flagged"]), set(lof["flagged"]), set(z["flagged"])
    consensus = sorted(i for i in range(len(df))
                       if (i in iso_set) + (i in lof_set) + (i in z_set) >= 2)

    flagged_rows = []
    for i in consensus:
        row = df.iloc[i]
        methods = []
        if i in iso_set: methods.append("IsolationForest")
        if i in lof_set: methods.append("LOF")
        if i in z_set: methods.append("Z-score")
        flagged_rows.append({
            "date": str(row["date"].date()),
            "entries": int(row["entries"]),
            "alarms": int(row["alarms"]),
            "alarm_rate": round(float(row["alarm_rate"]), 3),
            "entries_dev_ratio": round(float(row["entries_dev_ratio"]), 2),
            "alarm_dev_ratio": round(float(row["alarm_dev_ratio"]), 2),
            "entries_z_score": round(float(row["entries_z_score"]), 2),
            "flagged_by": methods
        })

    print(f"   IF: {len(iso['flagged'])}, LOF: {len(lof['flagged'])}, Z: {len(z['flagged'])}, consensus: {len(consensus)}")

    llm = ChatOllama(model="phi4-mini-reasoning", temperature=0)
    prompt = f"""Airport: {state['perimeter']}. Dataset: {len(df)} daily observations.

Three unsupervised methods flagged anomalies (IsolationForest, LOF, Z-score). These rows were flagged by at least two of them:

{json.dumps(flagged_rows, indent=2)}

Write a short analytical note (max 6 sentences) covering:
- which observations are the most clearly anomalous and why
- whether the anomalies cluster in time or look isolated
- which metric (alarm_rate, entries, deviation ratios) is driving most of the flags

Be specific with dates and numbers. No speculation, no bullet points."""

    analysis = llm.invoke([HumanMessage(content=prompt)]).content

    result = {
        "airport": state["perimeter"],
        "total_rows": len(df),
        "method_counts": {"IF": len(iso["flagged"]), "LOF": len(lof["flagged"]), "Z": len(z["flagged"])},
        "consensus_count": len(consensus),
        "consensus_indices": consensus,
        "flagged_rows": flagged_rows,
        "analysis": analysis
    }

    return {
        "anomaly_results": json.dumps(result),
        "messages": [HumanMessage(content=f"Outlier detection: {len(consensus)} consensus anomalies out of {len(df)} rows")]
    }


def risk_profiling_agent_node(state):
    print("RISK PROFILING AGENT")

    anomaly_data = json.loads(state["anomaly_results"])

    if "error" in anomaly_data:
        return {
            "risk_profile": json.dumps({"error": anomaly_data["error"]}),
            "messages": [HumanMessage(content="Risk profiling skipped")]
        }

    flagged = anomaly_data.get("flagged_rows", [])
    if not flagged:
        return {
            "risk_profile": json.dumps({"airport": state["perimeter"], "assessments": []}),
            "messages": [HumanMessage(content="No flagged rows to assess")]
        }

    llm = ChatOllama(model="qwen2.5:7b", temperature=0)

    prompt = f"""You are profiling risk for airport {state['perimeter']}. 
For each observation below, apply these four indicators:

- FLAG_RATE_SPIKE: alarm_dev_ratio > 3
- ENTRIES_SPIKE: entries_dev_ratio > 3
- SEASONAL_DEVIATION: abs(entries_z_score) > 2
- HIGH_ALARM_RATE: alarm_rate > 0.5

Assign risk_level:
- CRITICAL if 3 or 4 indicators trigger
- HIGH if 2 indicators trigger
- MODERATE if 1 indicator triggers
- LOW if 0 indicators trigger

Observations:
{json.dumps(flagged, indent=2)}

Return ONLY a JSON array, no markdown, no explanation before or after. Each element must have:
"date", "risk_level", "triggered": [list of indicator names], "note": one sentence in plain language."""

    raw = llm.invoke([HumanMessage(content=prompt)]).content

    try:
        assessments = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        if match:
            try:
                assessments = json.loads(match.group())
            except json.JSONDecodeError:
                assessments = [{"raw": raw}]
        else:
            assessments = [{"raw": raw}]

    counts = {}
    for a in assessments:
        level = a.get("risk_level", "UNKNOWN")
        counts[level] = counts.get(level, 0) + 1

    print(f"   Risk distribution: {counts}")

    profile = {
        "airport": state["perimeter"],
        "total_assessed": len(assessments),
        "distribution": counts,
        "assessments": assessments
    }

    return {
        "risk_profile": json.dumps(profile),
        "messages": [HumanMessage(content=f"Risk profile: {counts}")]
    }

## 7. The Workflow Graph
Using `StateGraph`, we define the "brain" of the operation. The workflow is a Directed Acyclic Graph (DAG) that ensures the **Data Agent** completes its task before passing the context to the **Baseline Agent**. We also integrate `MemorySaver` to allow for threaded conversations.

In [56]:
builder = StateGraph(AgentState)
builder.add_node("data_agent", data_agent_node)
builder.add_node("baseline_agent", baseline_agent_node)
builder.add_node("outlier_detection_agent", outlier_detection_agent_node)
builder.add_node("risk_profiling_agent", risk_profiling_agent_node)

builder.set_entry_point("data_agent")
builder.add_edge("data_agent", "baseline_agent")
builder.add_edge("baseline_agent", "outlier_detection_agent")
builder.add_edge("outlier_detection_agent", "risk_profiling_agent")
builder.add_edge("risk_profiling_agent", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)



## 8. Executing the Security Analysis
This is the final execution block. The user identifies a perimeter, and the MAS generates a finalized report. 
**Note**: If you see a 100% alarm rate for low-volume perimeters (like LIN), the agent correctly identifies this as a "low entries" incident rather than a systemic failure, demonstrating the power of combined math and reasoning.

In [57]:
user_q = input("Identify target perimeter: ")
if user_q:
    config = {"configurable": {"thread_id": "v1"}}
    final_state = None
    for event in app.stream({"messages": [HumanMessage(content=user_q)]}, config, stream_mode="values"):
        final_state = event

    print("\n" + "="*60)
    print(f"PERIMETER: {final_state['perimeter']}")
    print("="*60)

    print("\n--- OUTLIER DETECTION RESULTS ---")
    anomaly = json.loads(final_state["anomaly_results"])
    if "error" in anomaly:
        print(f"Error: {anomaly['error']}")
    else:
        print(f"Total rows analyzed: {anomaly['total_rows']}")
        print(f"Method counts: {anomaly['method_counts']}")
        print(f"Consensus anomalies: {anomaly['consensus_count']}")
        print(f"\nLLM analysis:\n{anomaly['analysis']}")

    print("\n--- RISK PROFILING RESULTS ---")
    risk = json.loads(final_state["risk_profile"])
    if "error" in risk:
        print(f"Error: {risk['error']}")
    else:
        print(f"Total assessed: {risk['total_assessed']}")
        print(f"Distribution: {risk['distribution']}")
        print(f"\nAssessments:")
        for a in risk["assessments"]:
            print(json.dumps(a, indent=2))

DATA AGENT: FCO (882 records)
BASELINE AGENT: Preparing Data Hand-off for Outlier Agent
OUTLIER DETECTION AGENT
   IF: 4, LOF: 4, Z: 9, consensus: 4
RISK PROFILING AGENT
   Risk distribution: {'LOW': 1, 'CRITICAL': 1, 'HIGH': 1, 'MODERATE': 1}

PERIMETER: FCO

--- OUTLIER DETECTION RESULTS ---
Total rows analyzed: 73
Method counts: {'IF': 4, 'LOF': 4, 'Z': 9}
Consensus anomalies: 4

LLM analysis:
<think>
Okay, let's tackle this analytical note. First, I need to identify the most clearly anomalous observations based on the dataset provided. There are four rows flagged by all three methods: Isolation Forest, LOF, and Z-score.

Looking at each row's metrics:

1. The first anomaly on 2024-01-02 has high entries (214) and alarms (40), with alarm_rate at ~18.7%. The dev ratios for entries and alarms are quite high (1.14 and 2.32 respectively), and the Z-score is 0.17, which is moderately unusual but not extremely so unless compared to historical norms.

2. The second on 2024-01-06 has zero e

In [ ]:
#user_q = input("Identify target perimeter: ")
#if user_q:
    #config = {"configurable": {"thread_id": "v1"}}
    #for event in app.stream({"messages": [HumanMessage(content=user_q)]}, config, stream_mode="values"):
        #final_state = event
    
    #print(f"\n🛡️ OFFICIAL REPORT:\n{final_state['messages'][-1].content}")
    #print("STATO FINALE:", final_state.keys())
    #print("PERIMETER:", final_state.get("perimeter"))

DATA AGENT: FCO (882 records)
BASELINE AGENT: Preparing Data Hand-off for Outlier Agent

🛡️ OFFICIAL REPORT:
Baseline dataframe generated for FCO with 73 daily rows. Ready for Outlier Detection.
STATO FINALE: dict_keys(['messages', 'perimeter', 'passenger_json', 'alarm_json', 'baseline_dataframe_json'])
PERIMETER: FCO


In [ ]:
validation_result = validate_baseline_output(
    passenger_json=final_state["passenger_json"],
    alarm_json=final_state["alarm_json"]
)

=== BUILDING BASELINE DATAFRAME ===

[1] BASIC STRUCTURE
Shape: (73, 14)
Columns: ['date', 'entries', 'investigations', 'alarms', 'alarm_rate', 'inv_rate', 'entries_7_mean', 'alarm_rate_7_mean', 'entries_dev_ratio', 'alarm_dev_ratio', 'month', 'monthly_mean', 'monthly_std', 'entries_z_score']

Head:
        date  entries  investigations  alarms  alarm_rate   inv_rate  \
0 2023-12-31        3               3     0.0    0.000000   1.000000   
1 2024-01-01      344             246    19.0    0.055233   0.715116   
2 2024-01-02      214            5213    40.0    0.186916  24.359813   
3 2024-01-03       35               0     0.0    0.000000   0.000000   
4 2024-01-04        1               1     0.0    0.000000   1.000000   

   entries_7_mean  alarm_rate_7_mean  entries_dev_ratio  alarm_dev_ratio  \
0             3.0           0.000000           1.000000         0.000000   
1           173.5           0.027616           1.982709         2.000000   
2           187.0           0.080716  